# Human In the Loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

1. High-stakes operations requiring human approval (e.g. database writes, financial transactions).
2. Compliance workflows where human oversight is mandatory.
3. Long-running conversations where human feedback guides the agent.

In [47]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    """mock function to read an emailby its id."""
    return f"Email content got id:{email_id}"

def send_email_tool(recipient:str,subject:str,body:str)->str:
    """mock function to send an email."""
    return f"Email sent to:{recipient} with subject'{subject}'"

In [48]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
model= init_chat_model("llama-3.3-70b-versatile",
                       model_provider= "groq"
                       )


In [49]:
agent=create_agent(
    model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [50]:
config = {"configurable": {"thread_id": "test-approve"}}

# Step 1: Request
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)

In [51]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0fbdad5d-7dc5-4e18-ab69-42952f5480a5'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4ksseekev', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.061852858, 'completion_tokens_details': None, 'prompt_time': 0.033495072, 'prompt_tokens_details': None, 'queue_time': 0.161917545, 'total_time': 0.09534793}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f4b9b-22e2-7703-a585-a4d678ad2a97-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bo

In [53]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f" Result: {result['messages'][-1].content}")

⏸ Paused! Approving...
 Result: I see you want to send an email. The function call to send the email is already provided. If you want to read an email by its id, you would use the read_email_tool function.


In [54]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0fbdad5d-7dc5-4e18-ab69-42952f5480a5'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4ksseekev', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.061852858, 'completion_tokens_details': None, 'prompt_time': 0.033495072, 'prompt_tokens_details': None, 'queue_time': 0.161917545, 'total_time': 0.09534793}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f4b9b-22e2-7703-a585-a4d678ad2a97-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bo

### Editing

In [55]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    """mock function to read an emailby its id."""
    return f"Email content got id:{email_id}"

def send_email_tool(recipient:str,subject:str,body:str)->str:
    """mock function to send an email."""
    return f"Email sent to:{recipient} with subject'{subject}'"

agent=create_agent(
    model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [56]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request(with wrong info)
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Test' and body 'Hello'"
            )
        ]
    },
    config=config
)


In [57]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='ebd5ed06-27a0-4c9b-99ba-afe7e788737e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'pe0kxakgv', 'function': {'arguments': '{"body":"Hello","recipient":"john@test.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 307, 'total_tokens': 337, 'completion_time': 0.065765398, 'completion_tokens_details': None, 'prompt_time': 0.031038843, 'prompt_tokens_details': None, 'queue_time': 0.058927146, 'total_time': 0.096804241}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f4b9b-aa22-75c2-8b62-d2e9a4ef2b58-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', '

In [58]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",  # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )


⏸️ Paused! Editing...


In [59]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='ebd5ed06-27a0-4c9b-99ba-afe7e788737e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'pe0kxakgv', 'function': {'arguments': '{"body":"Hello","recipient":"john@test.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 307, 'total_tokens': 337, 'completion_time': 0.065765398, 'completion_tokens_details': None, 'prompt_time': 0.031038843, 'prompt_tokens_details': None, 'queue_time': 0.058927146, 'total_time': 0.096804241}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f4b9b-aa22-75c2-8b62-d2e9a4ef2b58-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args'